In [1]:
from mypackage.utilities import connect_to_db
from mypackage.mapping import reverse_combined_column_mapping
import pandas as pd
from mypackage.config import get_date_range,groups_backend,groups_frontend,bus_lines,groups_middle,get_test_date,bus_lines

# 设定日期范围
date_range=get_date_range()
# date_range=pd.date_range(start='2026-01-01', end='2026-01-31', freq='D').date

# 获取名称列表
conn,cur=connect_to_db()
expense_categorys_1=[
  'FYXM0012',
  'FYXM0014',
  'FYXM0016',
  'FYXM0024',
  'FYXM0025',
  'FYXM0026',
  'FYXM0027',
  'FYXM0028',
  'FYXM0029',
  'FYXM0031',
  'FYXM0072',
  'FYXM0073',
]
expense_categorys_text_1=','.join([f"'{item}'" for item in expense_categorys_1])
expense_categorys_2=[
  'FYXM0012',
  'FYXM0014',
  'FYXM0016',
]
expense_categorys_text_2=','.join([f"'{item}'" for item in expense_categorys_2])

cur.execute(
  f"SELECT * FROM fact_expense WHERE (exp_item_code in({expense_categorys_text_1}) AND unique_lvl like '%行政中心%') OR (exp_item_code in({expense_categorys_text_2}) AND unique_lvl like '%人力资源中心%')"
)
df_expense=cur.fetchall()
columns = [reverse_combined_column_mapping.get(desc[0]) for desc in cur.description]
df_expense=pd.DataFrame(df_expense,columns=columns)
print(df_expense['会计期间'].drop_duplicates().sort_values().tolist())
# df_expense.to_csv('.//1-中后台数据拆分//1-df_expense.csv',index=False,encoding='utf-8-sig')
df_expense=df_expense[df_expense['会计期间'].isin(date_range)]
df_expense=df_expense[~df_expense['分摊业务线'].isin(bus_lines)]
# 已经计算过的费用
id_exclude=df_expense['来源编号'].drop_duplicates().tolist()
print(len(id_exclude))
df_expense.to_csv('.//1-中后台数据拆分//1-df_expense.csv',index=False,encoding='utf-8-sig')

# 导入组织架构的数据，用于将预分配层级映射为业务线
cur.execute(
  f"SELECT * FROM dim_org_struc"
)
columns = [reverse_combined_column_mapping.get(desc[0]) for desc in cur.description]
df_org=pd.DataFrame(cur.fetchall(),columns=columns)
# 关闭数据库连接
conn.close()

[datetime.date(2022, 1, 1), datetime.date(2022, 2, 1), datetime.date(2022, 3, 1), datetime.date(2022, 4, 1), datetime.date(2022, 5, 1), datetime.date(2022, 6, 1), datetime.date(2022, 7, 1), datetime.date(2022, 8, 1), datetime.date(2022, 9, 1), datetime.date(2022, 10, 1), datetime.date(2022, 11, 1), datetime.date(2022, 12, 1), datetime.date(2023, 1, 1), datetime.date(2023, 2, 1), datetime.date(2023, 3, 1), datetime.date(2023, 4, 1), datetime.date(2023, 5, 1), datetime.date(2023, 6, 1), datetime.date(2023, 7, 1), datetime.date(2023, 8, 1), datetime.date(2023, 9, 1), datetime.date(2023, 10, 1), datetime.date(2023, 11, 1), datetime.date(2023, 12, 1), datetime.date(2024, 1, 1), datetime.date(2024, 2, 1), datetime.date(2024, 3, 1), datetime.date(2024, 4, 1), datetime.date(2024, 5, 1), datetime.date(2024, 6, 1), datetime.date(2024, 7, 1), datetime.date(2024, 8, 1), datetime.date(2024, 9, 1), datetime.date(2024, 10, 1), datetime.date(2024, 11, 1), datetime.date(2024, 12, 1), datetime.date(2025

In [2]:
# 计算人数比例
from mypackage.utilities import connect_to_db
from mypackage.mapping import reverse_combined_column_mapping
import pandas as pd
import os
from mypackage.config import get_date_range
# 获取名称列表
conn,cur=connect_to_db()
cur.execute(
    '''
    SELECT * 
    FROM fact_personnel 
    WHERE 
        (unique_lvl NOT LIKE '%分公司%' OR third_org in ('增值业务产品部','分公司运营支持部','通道合作部','银行合作部'))
        AND unique_lvl NOT LIKE '%AGI%'
        AND unique_lvl NOT LIKE '%审核能力中心%' OR unique_lvl LIKE '%审核能力中心-运营中心%'
        AND class ='发薪人数'
    '''
)
df_rate=cur.fetchall()
columns = [reverse_combined_column_mapping.get(desc[0]) for desc in cur.description]
df_rate=pd.DataFrame(df_rate,columns=columns)
df_rate=df_rate[df_rate['日期'].isin(date_range)]
df_rate_group=df_rate.groupby(['唯一层级','日期']).agg({'人数':'sum'}).reset_index()
# 计算每个月的总人数
df_rate_group['月总人数'] = df_rate_group.groupby('日期')['人数'].transform('sum')
# 计算每条记录的人数比重
df_rate_group['比重'] = df_rate_group['人数'] / df_rate_group['月总人数']

df_rate_group.to_csv('.//1-中后台数据拆分//2-df_rate_group.csv',index=False,encoding='utf-8-sig')

df_rate_group.rename(columns={'日期':'会计期间'},inplace=True)
df_rate_group=df_rate_group[['会计期间','唯一层级','比重']]
conn.close()

In [3]:
# 将第一次分配的费用分摊到业务线上
df_merged = df_expense.merge(df_rate_group, on=['会计期间'], how='left')
df_merged.astype({'比重':'float','费用金额':'float'})
df_merged['费用金额']=df_merged['费用金额']*df_merged['比重']
df_merged=df_merged.drop(['比重'],axis=1).rename(columns={'唯一层级_y':'唯一层级','唯一层级_x':'数据来源'})
df_merged.to_csv('.//1-中后台数据拆分//3-df_merged.csv', index=False,encoding='utf-8-sig')

In [4]:
import numpy as np
# 拼接上中台自身的数据  
conn, cur = connect_to_db()
# 读取数据
cur.execute("SELECT * FROM fact_expense WHERE unique_lvl NOT LIKE '%无归属%'")
columns = [reverse_combined_column_mapping.get(
    desc[0]) for desc in cur.description]
data = cur.fetchall()
df = pd.DataFrame(data, columns=columns)
df=df[df['会计期间'].isin(date_range)]
df=df[~df['来源编号'].isin(id_exclude)]
df['数据来源']=df['唯一层级']
# 中台拼接自己的费用
df_template = pd.concat([df, df_merged])
df_template = df_template.reset_index()[
    ['来源编号', '唯一层级','单据编号','报销人', '摘要', '会计期间', '费用性质', '费用大类','核算项目-费控', '研发项目', '项目编码', '费用金额','数据来源','分摊业务线']]
df_template[['一级组织', '二级组织', '三级组织']
            ] = df_template['唯一层级'].str.split('-', n=2, expand=True)
df_template['会计期间'] = pd.to_datetime(df_template['会计期间'])
df_template['年份'] = df_template['会计期间'].dt.year
df_template['会计期间'] = df_template['会计期间'].dt.date
df_template = df_template[['来源编号', '唯一层级', '一级组织', '二级组织',
                            '三级组织','单据编号','报销人', '摘要', '会计期间', '费用性质', '费用大类', '核算项目-费控','研发项目', '项目编码', '费用金额', '年份','数据来源','分摊业务线']]
columns = bus_lines
df_template[columns] = np.nan

# 步骤1：先移除空值
df_template = df_template.dropna(subset=['费用金额'])
# 步骤2：再过滤掉0值
df_template = df_template[df_template['费用金额'] != 0]

df_template.rename(columns={'费用金额':'金额'},inplace=True)
# 根据组织架构映射表，将分摊层级调整为对应的业务线
# df_template = pd.merge(df_template, df_org[['唯一层级', '业务线']], left_on='分摊层级',right_on='唯一层级', how='left')
for col in columns:
    df_template[col] = np.where(df_template['分摊业务线'] == col, 1, np.nan)
# df_template=df_template.drop(['业务线','唯一层级_y'],axis=1).rename(columns={'唯一层级_x':'唯一层级'})
df_template.to_csv('.//1-中后台数据拆分//4-df_template.csv', index=False,encoding='utf-8-sig')

In [5]:
# 中台自身数据拆分
from mypackage.utilities import connect_to_db, business_line_split,organize_and_split_bus_line
import pandas as pd
import os
import shutil
from mypackage.config import get_filename,get_foldername

# 获取名称列表
conn, cur = connect_to_db()
errored_files = []
try:
    cur.execute("SELECT distinct unique_lvl, prim_org, sec_org,'third_org', short_name, category FROM dim_org_struc")
    df_path = pd.DataFrame(cur.fetchall(), columns=['unique_lvl', 'prim_org', 'sec_org','third_org', 'short_name', 'category'])
    df_path = df_path[df_path.category.isin(['中台'])]
    df=df_template
    df_org=df_path
    table_name='业务线费用拆分表'
    sheet_name='部门金额'
    date_column='会计期间'
    file_name=get_filename('业务线费用拆分表')
    groups=groups_middle
    folder_name=get_foldername()
    # 筛选df_org中尚未拆分的层级
    is_by_df=False
    is_split_others=False

    organize_and_split_bus_line(df, df_org, groups, date_range, date_column, table_name, sheet_name,folder_name,file_name,is_split_others,is_by_df)
    
finally:
    conn.close()



Z:\11-业务报表\8.业务线核算\国际渠道事业群\运营中心\2026年02月\业务线费用拆分表202602.xlsx
数据已成功写入 Excel 文件。
Z:\11-业务报表\8.业务线核算\支付硬件事业群\技术中心\2026年02月\业务线费用拆分表202602.xlsx
数据已成功写入 Excel 文件。
Z:\11-业务报表\8.业务线核算\支付服务事业群\培训中心\2026年02月\业务线费用拆分表202602.xlsx
数据已成功写入 Excel 文件。
Z:\11-业务报表\8.业务线核算\支付服务事业群\技术中心\2026年02月\业务线费用拆分表202602.xlsx
数据已成功写入 Excel 文件。
Z:\11-业务报表\8.业务线核算\国际渠道事业群\产品中心\2026年02月\业务线费用拆分表202602.xlsx
数据已成功写入 Excel 文件。
Z:\11-业务报表\8.业务线核算\支付服务事业群\监管事务中心\2026年02月\业务线费用拆分表202602.xlsx
数据已成功写入 Excel 文件。
Z:\11-业务报表\8.业务线核算\支付硬件事业群\集采中心\2026年02月\业务线费用拆分表202602.xlsx
数据已成功写入 Excel 文件。
Z:\11-业务报表\8.业务线核算\智造事业群\收单供应中心\2026年02月\业务线费用拆分表202602.xlsx
数据已成功写入 Excel 文件。
Z:\11-业务报表\8.业务线核算\支付服务事业群\支付安全中心\2026年02月\业务线费用拆分表202602.xlsx
数据已成功写入 Excel 文件。
Z:\11-业务报表\8.业务线核算\支付服务事业群\消保中心\2026年02月\业务线费用拆分表202602.xlsx
数据已成功写入 Excel 文件。
Z:\11-业务报表\8.业务线核算\支付硬件事业群\质量中心\2026年02月\业务线费用拆分表202602.xlsx
数据已成功写入 Excel 文件。
Z:\11-业务报表\8.业务线核算\支付服务事业群\产品中心\2026年02月\业务线费用拆分表202602.xlsx
数据已成功写入 Excel 文件。
Z:\11-业务报表\8.业务线核算\智造事业群\智造管理中心\2026年02月\业务线费用拆分

In [6]:
# 后台自身数据拆分（不含人力和房租等分摊费用）
from mypackage.utilities import connect_to_db, business_line_split,organize_and_split_bus_line
import pandas as pd
import os
import shutil
from mypackage.config import get_filename,get_foldername

# 获取名称列表
conn, cur = connect_to_db()
errored_files = []
try:
    cur.execute("SELECT distinct unique_lvl, prim_org, sec_org,third_org, short_name, category FROM dim_org_struc")
    df_path = pd.DataFrame(cur.fetchall(), columns=['unique_lvl', 'prim_org', 'sec_org', 'third_org','short_name', 'category'])
    df_path = df_path[df_path.category.isin(['后台'])]
    
    # 修正后的代码（明确创建副本，消除警告）
    df = df_template[(df_template['费用性质']!='人力费用') & ~(df_template['来源编号'].isin(id_exclude))].copy()

    # 此时，后续的赋值操作是安全的
    df.loc[:, '集团'] = np.nan
    df_org=df_path
    table_name='业务线费用拆分表'
    sheet_name='部门金额'
    date_column='会计期间'
    file_name=get_filename('业务线费用拆分表')
    groups=groups_backend
    folder_name=get_foldername()
    # 筛选df_org中尚未拆分的层级
    is_by_df=False
    is_split_others=True

    organize_and_split_bus_line(df, df_org, groups, date_range, date_column, table_name, sheet_name,folder_name,file_name,is_split_others,is_by_df)
    
finally:
    conn.close()


Z:\11-业务报表\8.业务线核算\董事会\董事会秘书处-证券部\2026年02月\业务线费用拆分表202602.xlsx
数据已成功写入 Excel 文件。
Z:\11-业务报表\8.业务线核算\资产运营中心\资产运营部\2026年02月\业务线费用拆分表202602.xlsx
数据已成功写入 Excel 文件。
Z:\11-业务报表\8.业务线核算\资产运营中心\在建项目组\2026年02月\业务线费用拆分表202602.xlsx
数据已成功写入 Excel 文件。
Z:\11-业务报表\8.业务线核算\公共事务中心\政府事务中心\2026年02月\业务线费用拆分表202602.xlsx
数据已成功写入 Excel 文件。
Z:\11-业务报表\8.业务线核算\董事会\董事会秘书处-战略发展部\2026年02月\业务线费用拆分表202602.xlsx
数据已成功写入 Excel 文件。
Z:\11-业务报表\8.业务线核算\董事会\审计监察部\2026年02月\业务线费用拆分表202602.xlsx
数据已成功写入 Excel 文件。
Z:\11-业务报表\8.业务线核算\董事会\董事会秘书处-品牌管理部\2026年02月\业务线费用拆分表202602.xlsx
数据已成功写入 Excel 文件。
Z:\11-业务报表\8.业务线核算\服务事业群\信息管理中心\2026年02月\业务线费用拆分表202602.xlsx
数据已成功写入 Excel 文件。
Z:\11-业务报表\8.业务线核算\服务事业群\人力资源中心\2026年02月\业务线费用拆分表202602.xlsx
数据已成功写入 Excel 文件。
Z:\11-业务报表\8.业务线核算\服务事业群\行政中心\2026年02月\业务线费用拆分表202602.xlsx
数据已成功写入 Excel 文件。
Z:\11-业务报表\8.业务线核算\服务事业群\法务中心\2026年02月\业务线费用拆分表202602.xlsx
数据已成功写入 Excel 文件。
剩余的层级放到一个文件中
剩余的层级： ['董事会-江汉-公共部门', '董事会-公共部门-挂靠人员', '服务事业群-其他-公共部门', '服务事业群-公共部门-公共部门', '公共事务中心-公共部门-公共部门', '计划财务中心-支付财务部-公共部门

In [7]:
# 后台自身数据拆分（含人力和分摊的部分）
from mypackage.utilities import connect_to_db, business_line_split,organize_and_split_bus_line,read_data_bysql
import pandas as pd
import os
import shutil
from mypackage.config import get_filename,get_foldername,bus_lines
# 获取名称列表
conn, cur = connect_to_db()
errored_files = []

# 获取预算人力的比例
sql="SELECT unique_lvl, bus_line, rate, date FROM fact_bus_wage_rate"
df_labor=read_data_bysql(sql)
df_labor['日期'] = pd.to_datetime(df_labor['日期'])
df_labor['日期']=(df_labor['日期']).dt.date
df_labor=df_labor[df_labor['日期'].isin(date_range)]
df_labor.rename(columns={'日期':'会计期间'},inplace=True)

try:
    cur.execute("SELECT distinct unique_lvl, prim_org, sec_org,third_org, short_name, category FROM dim_org_struc")
    df_path = pd.DataFrame(cur.fetchall(), columns=['unique_lvl', 'prim_org', 'sec_org', 'third_org','short_name', 'category'])
    df_path = df_path[df_path.category.isin(['后台'])]
    
    # 将预算的工资比例计算人力费用，并填入模板中
    df_1=df_template[(df_template['费用性质']=='人力费用') | (df_template['来源编号'].isin(id_exclude))].melt(
      id_vars=['来源编号', '唯一层级', '一级组织', '二级组织', '三级组织','单据编号', '报销人', '摘要', '会计期间', '费用性质',
       '费用大类', '核算项目-费控', '研发项目', '项目编码', '金额', '年份', '数据来源', '分摊业务线'],
      var_name='业务线',value_name='比例').drop('比例',axis=1)
    df_a=pd.merge(df_1,df_labor,how='left',on=['唯一层级','业务线','会计期间'])
    df_a.to_csv('df_a.csv',index=False,encoding='utf-8-sig')
    df=df_a.pivot(columns='业务线',values='比例',index=['来源编号', '唯一层级', '一级组织', '二级组织', '三级组织', '单据编号','报销人', '摘要', '会计期间', '费用性质',
          '费用大类', '核算项目-费控', '研发项目', '项目编码', '金额', '年份', '数据来源', '分摊业务线']).reset_index()
    df = df[df_template.columns]
    df=df.sort_values(['费用性质','唯一层级','来源编号'])
    
    # 填充分摊业务线的值
    for col in bus_lines:
        df[col] = np.where(df['分摊业务线'] == col, 1, df[col])

    df_org=df_path
    table_name='业务线费用拆分表'
    sheet_name='部门金额'
    date_column='会计期间'
    file_name=get_filename('业务线费用拆分表')
    # 全部后台的人力费用统一放入<其他-后台人力费用>文件夹
    groups=[]
    folder_name=get_foldername()
    # 筛选df_org中尚未拆分的层级
    is_by_df=False
    is_split_others=True
    is_labor=True

    organize_and_split_bus_line(df, df_org, groups, date_range, date_column, table_name, sheet_name,folder_name,file_name,is_split_others,is_by_df,is_labor)
    
finally:
    conn.close()

剩余的层级放到一个文件中
剩余的层级： ['董事会-江汉-公共部门', '服务事业群-行政中心-资产管理部', '董事会-董事会秘书处-证券部', '服务事业群-人力资源中心-招聘配置部', '董事会-公共部门-挂靠人员', '服务事业群-其他-公共部门', '服务事业群-公共部门-公共部门', '公共事务中心-公共部门-公共部门', '资产运营中心-在建项目组-公共部门', '资产运营中心-资产运营部-公共部门', '服务事业群-信息管理中心-基础架构与网络安全部', '服务事业群-人力资源中心-人事服务部', '计划财务中心-支付财务部-公共部门', '公共事务中心-政府事务中心-项目支持部', '服务事业群-人力资源中心-公共部门', '公共事务中心-政府事务中心-公共部门', '计划财务中心-财务管理部-公共部门', '服务事业群-行政中心-基础建设部', '董事会-董事会秘书处-公共部门', '董事会-董事会秘书处-投融资部', '服务事业群-行政中心-后勤服务部', '董事会-刘亚-公共部门', '服务事业群-人力资源中心-中正人力资源部', '服务事业群-法务中心-涉外法律事务部', '董事会-董事会秘书处-品牌管理部-品牌宣传部', '董事会-战略投资委员会-投资顾问', '董事会-石晓冬-公共部门', '董事会-战略投资委员会-公共部门', '董事会-童卫东-公共部门', '资产运营中心-在建项目组-建设工程小组', '计划财务中心-税务管理部-公共部门', '服务事业群-信息管理中心-系统管理部', '计划财务中心-公共部门-公共部门', '董事会-审计监察部-公共部门', '计划财务中心-资金管理部-公共部门', '董事会-董事会秘书处-战略发展部-公共部门', '抵销数-抵销数-抵销数', '计划财务中心-会计部-公共部门', '服务事业群-信息管理中心-公共部门', '服务事业群-人力资源中心-长沙人力资源部', '服务事业群-法务中心-国内法律事务部', '服务事业群-行政中心-流程管理部', '服务事业群-人力资源中心-学习发展部', '董事会-董事会秘书处-品牌管理部-品牌策划部', '董事会-董事会秘书处-嘉联', '资产运营中心-在建项目组-采购小组', '新国都本部-公共部门-公共部门', '资产运营中心-在建项目组-项目事务小

In [8]:
# 将前台中的需要单独拆分的业务线拉出来
from mypackage.utilities import connect_to_db, business_line_split
import pandas as pd
import os
import shutil

# Excel 文件路径和工作表名
sheet_name = '部门金额'
groups = groups_frontend

# 获取名称列表
conn, cur = connect_to_db()
errored_files = []
try:
    cur.execute("SELECT distinct unique_lvl, prim_org, sec_org,third_org,short_name, category FROM dim_org_struc")
    df_path = pd.DataFrame(cur.fetchall(), columns=['unique_lvl', 'prim_org', 'sec_org', 'third_org','short_name', 'category'])
    df_path = df_path[df_path.category.isin(['前台'])]
    
    df=df_template
    df_org=df_path
    table_name='业务线费用拆分表'  
    sheet_name='部门金额' 
    date_column='会计期间'
    file_name=get_filename('业务线费用拆分表')
    groups=groups_frontend
    folder_name=get_foldername()
    # 不拆分剩余的层级
    is_split_others=False
    

    organize_and_split_bus_line(df, df_org, groups, date_range, date_column, table_name, sheet_name,folder_name,file_name,is_split_others)

finally:
    conn.close()


Z:\11-业务报表\8.业务线核算\国内渠道事业群\合伙人营销中心\2026年02月\业务线费用拆分表202602.xlsx
数据已成功写入 Excel 文件。
Z:\11-业务报表\8.业务线核算\支付服务事业群\审核能力中心\2026年02月\业务线费用拆分表202602.xlsx
数据已成功写入 Excel 文件。
Z:\11-业务报表\8.业务线核算\国内渠道事业群\运营中心\2026年02月\业务线费用拆分表202602.xlsx
数据已成功写入 Excel 文件。
Z:\11-业务报表\8.业务线核算\创新事业群\消费电子中心\2026年02月\业务线费用拆分表202602.xlsx
数据已成功写入 Excel 文件。
